# Modeling the Control Policy (When to Intervene)

**MODELING notebook — pre-generation gating, the dev-to-eval methodology, policy value, and ceiling recovery**

This is a **modeling** notebook: every section introduces one modeling choice and evaluates it.

Every section follows *Question → What we do → Figure/Table → Reading → Artifact → Caveat*.
Each code cell states what it does and each output is interpreted in the following
cell, so a reader with no access to the code can follow the reasoning. All numbers are
read from immutable artifacts in `results/` through `paper_lib`; missing optional
experiments print `PENDING` with their producer command instead of failing.

In [ ]:
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
import paper_lib as L

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 130)
pd.set_option("display.width", 200)
np.random.seed(42)

PRIMARY_RUN = "M4_intersection_dev_conditioned_continuous"
BLIND_RUN = "M4_intersection_dev_blind_continuous"
GATE_FEATURE_COLS = ["top1_faiss", "top5_mean_faiss", "top5_min_faiss", "top5_std_faiss",
                     "retrieval_margin", "top5_spread", "retrieval_overlap", "max_lift",
                     "mean_lift", "n_positive_lifts", "n_negative_lifts", "lift_conflict",
                     "evidence_density", "query_desc_len", "query_title_len"]
print("Project root:", L.ROOT)

## The control problem

Even a calibrated prior has per-ticket variance: some tickets are helped, some harmed. A **control
policy** decides, before generating, whether to apply feedback to a given ticket. The train/dev/eval
split is designed for exactly this: feedback is *built* from train, every configuration is
*developed* on dev, patterns that hold on dev are *modeled*, and the resulting policy is *tested once*
on eval. This notebook builds and evaluates that policy on dev; notebook 06 confirms on eval.

## Why a control policy can only help under some conditions

A policy replaces feedback with the baseline on tickets it closes. If always-on has a positive mean,
closing tickets removes benefit on average, so a policy helps only if it can rank the **sign** of the
per-ticket effect better than chance. Formally, a policy beats always-on when the area under its
prediction curve exceeds a crossover near 0.5; at exactly chance it is worse, because the average
effect is positive. Under ticket-only feedback, always-on is *negative*, so closing harmful tickets
helps and the policy has much more room. This asymmetry is the key to everything below.

### What can the policy observe before generation?

**What we do.** Only pre-generation features: retrieval confidence, margin, spread, overlap, lift statistics, evidence density, and query lengths. No reference reply, no generated answer.

**Artifact.** `src/gate/features.py; results/blend_eb/gate_features_train.parquet`

**Caveat.** Team/class identity is excluded so the policy is not dataset-specific.

*What this cell does.* Load the gate feature table and show summary statistics for the features available at decision time.

In [ ]:
feat = pd.read_parquet(L.RESULTS / "blend_eb" / "gate_features_train_M4_intersection.parquet")
cols = [c for c in GATE_FEATURE_COLS if c in feat.columns]
display(feat[cols].describe().T.round(3))

**Reading.** The features are all quantities a retrieval system knows before calling the generator:
how confident the baseline is, how separated the top candidates are, how much feedback evidence
exists, and how large the potential bonus is. Because none of them is team- or class-specific, a
policy learned on them can transfer across organizations.

### Does a policy help under ticket-only feedback?

**What we do.** Train a logistic policy on generated dev labels with out-of-fold probabilities and evaluate its policy value.

**Artifact.** `results/gate_pilot_blind/{gate_pilot.csv,policy.csv}`

**Caveat.** Pilot on dev; eval confirmation follows.

*What this cell does.* Show the out-of-fold AUC and, for each threshold, the mean generated delta under the policy, alongside always-on, never-on, and oracle.

In [ ]:
pilot = L.load_gate_pilot("blind"); policy = L.load_gate_pilot_policy("blind")
display(pilot[["run", "n", "oof_auc", "mean_delta_always_on", "mean_delta_oracle", "ceiling_recovery"]].round(4))
display(policy.round(4))
fig, ax = plt.subplots(figsize=(10, 5))
sns.lineplot(data=policy[policy["policy"].isin(["learned", "always_on", "oracle"])],
             x="threshold", y="mean_delta", hue="run", style="policy", marker="o", ax=ax)
ax.axhline(0, color="black", lw=1); ax.set(ylabel="Mean generated delta under policy", title="Policy value (ticket-only feedback)")
plt.tight_layout(); L.savefig("05_policy_blind", run_ids=[]); plt.show()

**Reading.** Under ticket-only feedback the policy clearly helps: always-on is strongly negative,
while opening only the upper portion of predicted tickets recovers a substantial part of the oracle
ceiling (roughly 0.6–0.7 in the earlier analysis). The AUC is modest (about 0.63–0.71), but because
the alternative is strongly negative, even a modest ranking is valuable. This is the deployment
setting where the control policy earns its place.

### Does a policy help under resolution-informed feedback?

**What we do.** Repeat the same OOF policy evaluation on the resolution-informed runs.

**Artifact.** `results/gate_pilot_conditioned/gate_pilot.csv`

**Caveat.** Same features and label; only the feedback protocol changes.

*What this cell does.* Print the conditioned policy summary and compare its AUC and ceiling recovery to the blind case.

In [ ]:
cond = L.load_gate_pilot("conditioned")
display(cond[["run", "n", "oof_auc", "mean_delta_always_on", "mean_delta_oracle", "ceiling_recovery"]].round(4))
focus = pd.concat([pilot.assign(protocol="blind"), cond.assign(protocol="conditioned")])
fig, ax = plt.subplots(figsize=(10, 4.8))
sns.barplot(data=focus, x="run", y="oof_auc", hue="protocol", ax=ax)
ax.axhline(0.5, color="black", ls="--", lw=1); ax.set(ylabel="Out-of-fold AUC", title="Policy predictability by protocol")
plt.tight_layout(); L.savefig("05_auc_by_protocol", run_ids=[]); plt.show()

**Reading.** Under resolution-informed feedback the AUC is near chance (about 0.53–0.56) and the
policy cannot beat always-on. There is nothing to fix: the prior is already well-behaved and the
remaining per-ticket variance is not predictable from pre-generation features. The honest conclusion
is that a control policy is justified only when feedback reliability is low.

### Is a learned policy worth it, or does a simple rule suffice?

**What we do.** Compare the best learned threshold policy against the best single-feature rule on the same tickets.

**Artifact.** `results/gate_pilot_*/rules.csv`

**Caveat.** Rules are interpretable; the comparison bounds the value of learning.

*What this cell does.* For each run, show the best learned policy value and the best simple rule value side by side.

In [ ]:
rows = []
for tag in ["blind", "conditioned"]:
    pol = L.load_gate_pilot_policy(tag); rl = L.load_gate_pilot_rules(tag)
    learned = pol[pol["policy"] == "learned"].groupby("run")["mean_delta"].max()
    rules = rl.groupby("run")["mean_delta"].max()
    for run in learned.index:
        rows.append({"run": run, "protocol": tag, "best_learned": learned[run],
                     "best_rule": rules.get(run, np.nan)})
cmp = pd.DataFrame(rows); display(cmp.round(4))

**Reading.** Under ticket-only feedback the learned policy beats the best single-feature rule,
justifying the classifier. Under resolution-informed feedback both are dominated by always-on, again
showing that no policy is needed there. The simple rule (open when baseline confidence is low) is a
useful interpretable fallback but is not sufficient on its own.

### Does calibration already do the policy's job?

**What we do.** Compare the policy value on the uncalibrated (legacy Laplace) prior against the calibrated prior.

**Artifact.** `results/gate_pilot_blind{,_eb}/gate_pilot.csv`

**Caveat.** Same features; different prior behind the feedback.

*What this cell does.* Print both blind policy summaries side by side.

In [ ]:
pl = L.load_gate_pilot("blind"); peb = L.load_gate_pilot("blind_eb")
both = pd.concat([pl.assign(prior="legacy Laplace"), peb.assign(prior="calibrated EB")])
display(both[["prior", "run", "oof_auc", "mean_delta_always_on", "mean_delta_oracle", "ceiling_recovery"]].round(4))

**Reading.** On the uncalibrated prior the policy recovers a large fraction of the ceiling, turning a
clearly negative always-on into a positive policy. On the calibrated prior, always-on is already
near-safe, so the policy's *additional* value is small even though its AUC is higher. Calibration
and control are therefore **substitutes**: either discipline the prior or gate a raw one. We
recommend calibration as the default (it needs no labels) and keep the policy for settings where the
prior cannot be trusted.

## Conclusion — the control policy

A pre-generation policy is valuable exactly when feedback is unreliable and always-on is harmful.
Under ticket-only feedback it turns a negative always-on into a positive policy and recovers most of
the oracle ceiling; under resolution-informed feedback it is unnecessary. The next notebook confirms
the resulting recommendation on the untouched eval split.